In [15]:
import base64
import os
import sys
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import snowflake.connector
from cryptography.hazmat.primitives import serialization
from dotenv import dotenv_values
from pyspark.sql import SparkSession, Window, functions as F, types as T

In [2]:
# Capture the processing date once for both feeds.
processing_timezone = "Asia/Kolkata"
partition_date = datetime.now(
    ZoneInfo(processing_timezone)
).strftime("%d%m%Y")

# Configure local Spark.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["JAVA_HOME"] = (
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
)
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 512m pyspark-shell"

spark = (
    SparkSession.builder
    .appName("SkyPointsStaging")
    .master("local[1]")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.sql.session.timeZone", processing_timezone)
    .config(
        "spark.jars.packages",
        "net.snowflake:spark-snowflake_2.12:3.2.0-spark_3.5,"
        "net.snowflake:snowflake-jdbc:4.0.2",
    )
    .getOrCreate()
)

# Read the existing credentials.
project_root = Path(
    "/Users/mskbaruah/code/Incubyte Take home assignment/skypoints-pipeline"
)
sf = dotenv_values(project_root / ".env.local", interpolate=False)

# Prepare the private key.
passphrase = sf.get("SNOWFLAKE_PRIVATE_KEY_PASSPHRASE")

private_key = serialization.load_pem_private_key(
    Path(sf["SNOWFLAKE_LOCAL_PRIVATE_KEY_FILE"]).read_bytes(),
    password=passphrase.encode() if passphrase else None,
)

key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)

# Spark connection options. Do not display this dictionary.
sf_options = {
    "sfURL": f"{sf['SNOWFLAKE_ACCOUNT']}.snowflakecomputing.com",
    "sfUser": sf["SNOWFLAKE_USER"],
    "sfRole": sf["SNOWFLAKE_ROLE"],
    "sfWarehouse": sf["SNOWFLAKE_WAREHOUSE"],
    "sfDatabase": sf["SNOWFLAKE_DATABASE"],
    "sfSchema": "RAW",
    "pem_private_key": base64.b64encode(key_bytes).decode("ascii"),
}

# Native SQL connection for control queries.
sf_connection = snowflake.connector.connect(
    account=sf["SNOWFLAKE_ACCOUNT"],
    user=sf["SNOWFLAKE_USER"],
    role=sf["SNOWFLAKE_ROLE"],
    warehouse=sf["SNOWFLAKE_WAREHOUSE"],
    database=sf["SNOWFLAKE_DATABASE"],
    schema="RAW",
    authenticator="SNOWFLAKE_JWT",
    private_key=key_bytes,
    session_parameters={"TIMEZONE": "UTC"},
)

with sf_connection.cursor() as cursor:
    cursor.execute("SELECT 1")
    cursor.fetchone()

print("Connected to Snowflake.")
print(f"Processing partition: {partition_date}")

26/11/23 09:51:52 WARN Utils: Your hostname, Monisankars-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.11 instead (on interface en0)
26/11/23 09:51:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/mskbaruah/.ivy2/cache
The jars for the packages stored in: /Users/mskbaruah/.ivy2/jars
net.snowflake#spark-snowflake_2.12 added as a dependency
net.snowflake#snowflake-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-06857760-28cc-43c6-af01-4ec2f68b330b;1.0
	confs: [default]
	found net.snowflake#spark-snowflake_2.12;3.2.0-spark_3.5 in central
	found net.snowflake#snowflake-jdbc;4.0.2 in central


:: loading settings :: url = jar:file:/Users/mskbaruah/code/Incubyte%20Take%20home%20assignment/skypoints-pipeline/skypoints/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.parquet#parquet-avro;1.15.2 in central
	found org.apache.parquet#parquet-column;1.15.2 in central
	found org.apache.parquet#parquet-common;1.15.2 in central
	found org.apache.parquet#parquet-format-structures;1.15.2 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.slf4j#slf4j-api;1.7.33 in central
	found org.apache.parquet#parquet-encoding;1.15.2 in central
	found org.apache.parquet#parquet-hadoop;1.15.2 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found io.airlift#aircompressor;2.0.2 in central
	found commons-pool#commons-pool;1.6 in central
	found com.github.luben#zstd-jni;1.5.6-6 in central
	found org.apache.avro#avro;1.11.4 in central
	found com.fasterxml.jackson.core#jackson-core;2.14.3 in central
	found com.fasterxml.jackson.core#jackson-databind;2.14.3 in central
	found com.fasterxml.jackson.core#jackson-annotations;2.14.3 in central
	found org.apache.commons#commons-compress;1.26.2 in central
	found co

Connected to Snowflake.
Processing partition: 23112026


In [6]:
members_query = f"""
    SELECT RAW_RECORD, PARTITION_DATE
    FROM RAW.MEMBERS_RAW
    WHERE PARTITION_DATE = '{partition_date}'
"""

redemptions_query = f"""
    SELECT RAW_PAYLOAD, PARTITION_DATE
    FROM RAW.REDEMPTIONS_RAW
    WHERE PARTITION_DATE = '{partition_date}'
"""

members_raw = (
    spark.read
    .format("net.snowflake.spark.snowflake")
    .options(**sf_options)
    .option("query", members_query)
    .load()
)

redemptions_raw = (
    spark.read
    .format("net.snowflake.spark.snowflake")
    .options(**sf_options)
    .option("query", redemptions_query)
    .load()
)

print(f"Partition: {partition_date}")
print(f"Member RAW rows: {members_raw.count():,}")
print(f"Redemption RAW rows: {redemptions_raw.count():,}")

Partition: 23112026


Member RAW rows: 10,001


Redemption RAW rows: 10,000


In [9]:
members_raw.show(5, truncate = False)
redemptions_raw.show(5, truncate = False)

+----------------------------------------------------------------------------------------------------------+--------------+
|RAW_RECORD                                                                                                |PARTITION_DATE|
+----------------------------------------------------------------------------------------------------------+--------------+
||H|Member_Name|Member_Id|Enrollment_Date|Last_Flight_Date|Tier_Code|Agent_Name|State|Country|DOB|Is_Active|23112026      |
||D|Monserrate Johns|1|20230211|20260922|SLV|Olivia Hart|03|LCA|04011995|I                                 |23112026      |
||D|Hector Heller|2|20231002|20260708|BLU|Maya Arden|02|ARE|13061978|A                                     |23112026      |
||D|Keshaun Lang|3|20180619|20260830|SLV|Leah Nolan|02|PRT|07041995|I                                      |23112026      |
||D|Carmela Harvey|4|20231212|20260720|BLU|Kiran Ellis|11|WSM|20011985|A                                   |23112026      |
+-------

In [10]:
member_schema = T.StructType([
    T.StructField("MEMBER_NAME", T.StringType(), True),
    T.StructField("MEMBER_ID", T.StringType(), True),
    T.StructField("ENROLLMENT_DATE", T.StringType(), True),
    T.StructField("LAST_FLIGHT_DATE", T.StringType(), True),
    T.StructField("TIER_CODE", T.StringType(), True),
    T.StructField("AGENT_NAME", T.StringType(), True),
    T.StructField("STATE", T.StringType(), True),
    T.StructField("COUNTRY", T.StringType(), True),
    T.StructField("DOB", T.StringType(), True),
    T.StructField("IS_ACTIVE", T.StringType(), True),
])

transaction_schema = T.StructType([
    T.StructField("txn_id", T.StringType(), True),
    T.StructField("txn_date", T.StringType(), True),
    T.StructField("partner", T.StringType(), True),
    T.StructField("miles_redeemed", T.LongType(), True),
    T.StructField("status", T.StringType(), True),
])

redemption_schema = T.StructType([
    T.StructField("member_id", T.StringType(), True),
    T.StructField("feed_date", T.StringType(), True),
    T.StructField(
        "redemptions",
        T.ArrayType(transaction_schema),
        True,
    ),
    T.StructField("_corrupt_record", T.StringType(), True),
])

In [11]:
members_split = members_raw.withColumn(
    "_FIELDS",
    F.split(F.col("RAW_RECORD"), r"\|", -1),
)

is_header = F.col("RAW_RECORD").startswith("|H|")

# Leading empty value + record marker + 10 source fields.
is_detail = (
    F.col("RAW_RECORD").startswith("|D|")
    & (F.size("_FIELDS") == 12)
)

# Keep malformed records available for subsequent validation.
members_parse_errors = (
    members_split
    .filter(~F.coalesce(is_header | is_detail, F.lit(False)))
    .select("RAW_RECORD", "PARTITION_DATE")
)

members_df = (
    members_split
    .filter(is_detail)
    .select(
        *[
            F.col("_FIELDS")
            .getItem(position + 2)
            .cast(field.dataType)
            .alias(field.name)
            for position, field in enumerate(member_schema.fields)
        ],

        # Post Code is absent from the supplied source layout.
        F.lit(None).cast("int").alias("POST_CODE"),
        F.col("PARTITION_DATE"),
    )
)

members_df.show(5, truncate=False)

+----------------+---------+---------------+----------------+---------+-----------+-----+-------+--------+---------+---------+--------------+
|MEMBER_NAME     |MEMBER_ID|ENROLLMENT_DATE|LAST_FLIGHT_DATE|TIER_CODE|AGENT_NAME |STATE|COUNTRY|DOB     |IS_ACTIVE|POST_CODE|PARTITION_DATE|
+----------------+---------+---------------+----------------+---------+-----------+-----+-------+--------+---------+---------+--------------+
|Monserrate Johns|1        |20230211       |20260922        |SLV      |Olivia Hart|03   |LCA    |04011995|I        |NULL     |23112026      |
|Hector Heller   |2        |20231002       |20260708        |BLU      |Maya Arden |02   |ARE    |13061978|A        |NULL     |23112026      |
|Keshaun Lang    |3        |20180619       |20260830        |SLV      |Leah Nolan |02   |PRT    |07041995|I        |NULL     |23112026      |
|Carmela Harvey  |4        |20231212       |20260720        |BLU      |Kiran Ellis|11   |WSM    |20011985|A        |NULL     |23112026      |
|Alyci

In [12]:
redemptions_parsed = redemptions_raw.withColumn(
    "PAYLOAD",
    F.from_json(
        F.col("RAW_PAYLOAD"),
        redemption_schema,
        options={
            "mode": "PERMISSIVE",
            "columnNameOfCorruptRecord": "_corrupt_record",
        },
    ),
)

has_expected_structure = (
    F.col("PAYLOAD").isNotNull()
    & F.col("PAYLOAD._corrupt_record").isNull()
    & F.col("PAYLOAD.redemptions").isNotNull()
)

redemptions_parse_errors = (
    redemptions_parsed
    .filter(~has_expected_structure)
    .select("RAW_PAYLOAD", "PARTITION_DATE")
)

redemptions_df = (
    redemptions_parsed
    .filter(has_expected_structure)
    .select(
        F.col("PAYLOAD.member_id").alias("MEMBER_ID"),
        F.col("PAYLOAD.feed_date").alias("FEED_DATE"),
        F.explode("PAYLOAD.redemptions").alias("REDEMPTION"),
        F.col("PARTITION_DATE"),
    )
    .select(
        "MEMBER_ID",
        "FEED_DATE",
        F.col("REDEMPTION.txn_id").alias("TXN_ID"),
        F.col("REDEMPTION.txn_date").alias("TXN_DATE"),
        F.col("REDEMPTION.partner").alias("PARTNER"),
        F.col("REDEMPTION.miles_redeemed").alias("MILES_REDEEMED"),
        F.col("REDEMPTION.status").alias("STATUS"),
        "PARTITION_DATE",
    )
)

redemptions_df.show(5, truncate=False)

+---------+---------+------------------+--------+---------------------+--------------+---------+--------------+
|MEMBER_ID|FEED_DATE|TXN_ID            |TXN_DATE|PARTNER              |MILES_REDEEMED|STATUS   |PARTITION_DATE|
+---------+---------+------------------+--------+---------------------+--------------+---------+--------------+
|1        |20261121 |RX0000000000000001|20250206|LoungeHarbor Services|4000          |PENDING  |23112026      |
|1        |20261121 |RX0000000000000002|20230606|MarketBridge Rewards |9000          |COMPLETED|23112026      |
|1        |20261121 |RX0000000000000003|20250614|TripBridge Travel    |85000         |COMPLETED|23112026      |
|1        |20261121 |RX0000000000000004|20260921|LoungeHarbor Services|4000          |PENDING  |23112026      |
|1        |20261121 |RX0000000000000005|20240512|LoungeHarbor Services|4000          |COMPLETED|23112026      |
+---------+---------+------------------+--------+---------------------+--------------+---------+--------

In [18]:
#validation functions
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")


def clean_text(data):
    """Trim text and turn empty strings into null."""
    for column_name, data_type in data.dtypes:
        if data_type == "string":
            value = F.trim(F.col(column_name))

            data = data.withColumn(
                column_name,
                F.when(value == "", F.lit(None)).otherwise(value),
            )

    return data


def parse_date(column_name, date_format):
    """Parse an eight-digit date. Invalid dates become null."""
    value = F.col(column_name)

    return F.when(
        value.rlike(r"^\d{8}$"),
        F.try_to_timestamp(value, F.lit(date_format)).cast("date"),
    )


def validate_rows(data, rules):
    """Separate valid rows from rejected rows, keeping rejection reasons."""
    error_messages = [
        F.when(condition, message)
        for condition, message in rules
    ]

    checked = data.withColumn(
        "VALIDATION_ERRORS",
        F.concat_ws("; ", *error_messages),
    )

    valid = (
        checked
        .filter(F.col("VALIDATION_ERRORS") == "")
        .drop("VALIDATION_ERRORS")
    )

    rejected = checked.filter(F.col("VALIDATION_ERRORS") != "")

    return valid, rejected

In [19]:
#validate members data and convert the valid dates
members = clean_text(members_df)

enrollment_date = parse_date("ENROLLMENT_DATE", "yyyyMMdd")
last_flight_date = parse_date("LAST_FLIGHT_DATE", "yyyyMMdd")
birth_date = parse_date("DOB", "ddMMyyyy")

member_rules = [
    (
        F.col("MEMBER_NAME").isNull(),
        "MEMBER_NAME is required",
    ),
    (
        F.col("MEMBER_ID").isNull(),
        "MEMBER_ID is required",
    ),
    (
        enrollment_date.isNull(),
        "ENROLLMENT_DATE must be a valid yyyyMMdd date",
    ),
    (
        F.col("LAST_FLIGHT_DATE").isNotNull() & last_flight_date.isNull(),
        "LAST_FLIGHT_DATE must be a valid yyyyMMdd date",
    ),
    (
        F.col("DOB").isNotNull() & birth_date.isNull(),
        "DOB must be a valid ddMMyyyy date",
    ),
]

member_length_limits = {
    "MEMBER_NAME": 255,
    "MEMBER_ID": 18,
    "TIER_CODE": 5,
    "AGENT_NAME": 255,
    "STATE": 5,
    "COUNTRY": 5,
    "IS_ACTIVE": 1,
}

for column_name, limit in member_length_limits.items():
    member_rules.append(
        (
            F.length(F.col(column_name)) > limit,
            f"{column_name} exceeds {limit} characters",
        )
    )

members_valid, members_rejected = validate_rows(
    members,
    member_rules,
)

members_valid = (
    members_valid
    .withColumn("ENROLLMENT_DATE", enrollment_date)
    .withColumn("LAST_FLIGHT_DATE", last_flight_date)
    .withColumn("DOB", birth_date)
)

In [20]:
#validate redemption data and convert valid dates

redemptions = clean_text(redemptions_df)

feed_date = parse_date("FEED_DATE", "yyyyMMdd")
transaction_date = parse_date("TXN_DATE", "yyyyMMdd")

redemption_rules = [
    (
        F.col("MEMBER_ID").isNull(),
        "MEMBER_ID is required",
    ),
    (
        F.length(F.col("MEMBER_ID")) > 18,
        "MEMBER_ID exceeds 18 characters",
    ),
    (
        F.col("TXN_ID").isNull(),
        "TXN_ID is required",
    ),
    (
        F.col("PARTNER").isNull(),
        "PARTNER is required",
    ),
    (
        F.col("STATUS").isNull(),
        "STATUS is required",
    ),
    (
        feed_date.isNull(),
        "FEED_DATE must be a valid yyyyMMdd date",
    ),
    (
        transaction_date.isNull(),
        "TXN_DATE must be a valid yyyyMMdd date",
    ),
    (
        F.col("MILES_REDEEMED").isNull() | (F.col("MILES_REDEEMED") < 0),
        "MILES_REDEEMED must be a non-negative integer",
    ),
]

redemptions_valid, redemptions_rejected = validate_rows(
    redemptions,
    redemption_rules,
)

redemptions_valid = (
    redemptions_valid
    .withColumn("FEED_DATE", feed_date)
    .withColumn("TXN_DATE", transaction_date)
)

In [21]:
members_counted = (
    members_valid
    .groupBy(*members_valid.columns)
    .agg(F.count("*").alias("COPY_COUNT"))
)

members_duplicates = members_counted.filter(
    F.col("COPY_COUNT") > 1
)

members_unique = members_counted.drop("COPY_COUNT")


redemptions_counted = (
    redemptions_valid
    .groupBy(*redemptions_valid.columns)
    .agg(F.count("*").alias("COPY_COUNT"))
)

redemptions_duplicates = redemptions_counted.filter(
    F.col("COPY_COUNT") > 1
)

redemptions_unique = redemptions_counted.drop("COPY_COUNT")

In [22]:
# Check member keys within the selected partition.
member_name_window = Window.partitionBy(
    "PARTITION_DATE",
    "MEMBER_NAME",
)

member_id_window = Window.partitionBy(
    "PARTITION_DATE",
    "MEMBER_ID",
)

members_checked = (
    members_unique
    .withColumn("_NAME_COUNT", F.count("*").over(member_name_window))
    .withColumn("_ID_COUNT", F.count("*").over(member_id_window))
)

members_clean, members_conflicts = validate_rows(
    members_checked,
    [
        (
            F.col("_NAME_COUNT") > 1,
            "Different records share the same MEMBER_NAME",
        ),
        (
            F.col("_ID_COUNT") > 1,
            "Different records share the same MEMBER_ID",
        ),
    ],
)

members_clean = members_clean.drop("_NAME_COUNT", "_ID_COUNT")
members_conflicts = members_conflicts.drop("_NAME_COUNT", "_ID_COUNT")


# Check transaction keys within the selected partition.
transaction_window = Window.partitionBy(
    "PARTITION_DATE",
    "PARTNER",
    "TXN_ID",
)

redemptions_checked = redemptions_unique.withColumn(
    "_TRANSACTION_COUNT",
    F.count("*").over(transaction_window),
)

redemptions_clean, redemptions_conflicts = validate_rows(
    redemptions_checked,
    [
        (
            F.col("_TRANSACTION_COUNT") > 1,
            "Different records share the same PARTNER and TXN_ID",
        ),
    ],
)

redemptions_clean = redemptions_clean.drop("_TRANSACTION_COUNT")
redemptions_conflicts = redemptions_conflicts.drop("_TRANSACTION_COUNT")